# VarFlExI Time-Complexity Analysis

Ingests `results/time_complexity/benchmark_runs.csv`, `benchmark_summary.csv`, and per-run `memory_samples.csv` files. The fitted exponents are empirical scaling observations over the sampled range, not formal asymptotic claims.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_ROOT = Path('../results/time_complexity').resolve()
RUNS_CSV = RESULTS_ROOT / 'benchmark_runs.csv'
SUMMARY_CSV = RESULTS_ROOT / 'benchmark_summary.csv'
FIG_DIR = RESULTS_ROOT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

runs = pd.read_csv(RUNS_CSV)
summary = pd.read_csv(SUMMARY_CSV)
COMPONENTS = ['fluid_core', 'fluid_callback', 'solid', 'f_to_s', 's_to_f', 'io', 'other']
runs.head()

## Run-Level Data

In [ ]:
cols = ['sweep','parameter','parameter_value','run_id','dt','n_steps','n_surface','n_structural_dof','n_pperstep','n_particles','t_benchmark_s','wall_total_s','mean_step_s','m_peak_mib','phase_peak']
runs[[c for c in cols if c in runs.columns]].sort_values(['sweep','parameter_value','run_id'])

## Required Total and Per-Step Scaling Plots

In [ ]:
def plot_metric(sweep, xcol, ycol, ylabel, loglog=False):
    data = runs[runs['sweep'] == sweep].copy()
    if data.empty or xcol not in data or ycol not in data:
        return None
    grouped = data.groupby(xcol)[ycol].agg(['mean','std']).reset_index().sort_values(xcol)
    fig, ax = plt.subplots(figsize=(6.5,4.0))
    ax.errorbar(grouped[xcol], grouped['mean'], yerr=grouped['std'], marker='o', capsize=3)
    ax.set_xlabel(xcol)
    ax.set_ylabel(ylabel)
    if loglog:
        ax.set_xscale('log'); ax.set_yscale('log')
    ax.grid(True, which='both', alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'{sweep}_{ycol}_vs_{xcol}.png', dpi=200)
    return fig, ax

plot_metric('dt', 'dt', 't_benchmark_s', 'Post-warm-up measured solver-component time [s]')
plot_metric('dt', 'dt', 'mean_step_s', 'Mean post-warm-up step time [s]')
plot_metric('spatial', 'n_structural_dof', 't_benchmark_s', 'Post-warm-up measured solver-component time [s]', loglog=True)
plot_metric('spatial', 'n_structural_dof', 'mean_step_s', 'Mean post-warm-up step time [s]', loglog=True)
plot_metric('pperstep', 'n_particles', 't_benchmark_s', 'Post-warm-up measured solver-component time [s]', loglog=True)
plot_metric('pperstep', 'n_particles', 'mean_step_s', 'Mean post-warm-up step time [s]', loglog=True)

## Component Timings and Bottleneck Fractions

In [ ]:
def plot_components(sweep, xcol, per_step=False):
    data = runs[runs['sweep'] == sweep].copy()
    if data.empty:
        return None
    prefix = 'mean_' if per_step else 't_'
    suffix = '_step_s' if per_step else '_s'
    fig, ax = plt.subplots(figsize=(7.5,4.5))
    for comp in COMPONENTS:
        col = f'{prefix}{comp}{suffix}'
        if col not in data:
            continue
        grouped = data.groupby(xcol)[col].mean().reset_index().sort_values(xcol)
        ax.plot(grouped[xcol], grouped[col], marker='o', label=comp)
    ax.set_xlabel(xcol)
    ax.set_ylabel('Mean component time per step [s]' if per_step else 'Total component time [s]')
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2)
    fig.tight_layout()
    label = 'per_step' if per_step else 'total'
    fig.savefig(FIG_DIR / f'{sweep}_components_{label}.png', dpi=200)
    return fig, ax

plot_components('dt', 'dt', per_step=False)
plot_components('dt', 'dt', per_step=True)
plot_components('spatial', 'n_structural_dof', per_step=True)
plot_components('pperstep', 'n_particles', per_step=True)

frac_cols = [c for c in runs.columns if c.startswith('frac_')]
runs.groupby(['sweep','parameter_value'])[frac_cols].mean()

## Empirical Log-Log Scaling Exponents

In [ ]:
def fit_power_law(sweep, xcol, ycol):
    data = runs[runs['sweep'] == sweep].groupby(xcol)[ycol].mean().reset_index().replace([np.inf,-np.inf], np.nan).dropna()
    data = data[(data[xcol] > 0) & (data[ycol] > 0)]
    if len(data) < 2:
        return None
    p, b = np.polyfit(np.log(data[xcol]), np.log(data[ycol]), 1)
    return {'sweep': sweep, 'x': xcol, 'y': ycol, 'p': p, 'x_min': data[xcol].min(), 'x_max': data[xcol].max(), 'n': len(data)}

fits = []
for sweep, xcol in [('spatial','n_structural_dof'), ('pperstep','n_particles')]:
    for ycol in ['t_benchmark_s','mean_step_s'] + [f'mean_{c}_step_s' for c in COMPONENTS]:
        if ycol in runs.columns:
            fit = fit_power_law(sweep, xcol, ycol)
            if fit:
                fits.append(fit)
pd.DataFrame(fits)

## Peak Memory Analysis

In [ ]:
mem_cols = ['sweep','parameter_value','run_id','m_peak_mib','t_peak_s','n_step_peak','phase_peak']
runs[[c for c in mem_cols if c in runs.columns]].sort_values(['sweep','parameter_value','run_id'])

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
for sweep, data in runs.groupby('sweep'):
    ax.scatter(data['run_id'], data['m_peak_mib'], label=sweep, alpha=0.8)
ax.set_xlabel('run id')
ax.set_ylabel('Peak process-tree RSS [MiB]')
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'peak_memory_by_run.png', dpi=200)

In [ ]:
def plot_memory_trace(run_dir):
    path = Path(run_dir) / 'memory_samples.csv'
    if not path.exists():
        return None
    mem = pd.read_csv(path)
    if mem.empty:
        return None
    peak = mem.loc[mem['rss_mib'].idxmax()]
    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(mem['elapsed_s'], mem['rss_mib'])
    ax.scatter([peak['elapsed_s']], [peak['rss_mib']], color='red', zorder=3, label=f"peak step {int(peak['step'])}")
    ax.set_xlabel('elapsed wall time [s]')
    ax.set_ylabel('process-tree RSS [MiB]')
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    return fig, ax

plot_memory_trace(runs.iloc[0]['run_dir'])